# Batch re-encode a folder of videos

Runs `reencode_video` on every video in `INPUT_FOLDER`, writing `<name>.reencoded.mp4` next to each source (or into `OUTPUT_FOLDER`). Matches the `*.reencoded` naming the Lightning Pose / DLC pipeline expects.

In [1]:
import os
import shutil
import subprocess
import sys

def find_ffmpeg() -> str:
    """Locate ffmpeg: PATH, then the kernel's conda env, then the DEEPLABCUT env."""
    candidates = [
        shutil.which("ffmpeg"),
        os.path.join(sys.prefix, "Library", "bin", "ffmpeg.exe"),
        r"C:\Users\Almog\anaconda3\envs\DEEPLABCUT\Library\bin\ffmpeg.exe",
    ]
    for c in candidates:
        if c and os.path.isfile(c):
            return c
    raise FileNotFoundError("ffmpeg not found; install it (conda install -c conda-forge ffmpeg) or add it to PATH")

FFMPEG = find_ffmpeg()
print("using ffmpeg:", FFMPEG)

def reencode_video(input_file: str, output_file: str) -> None:
    """reencodes video into H.264 coded format using ffmpeg from a subprocess.

    Args:
        input_file: abspath to existing video
        output_file: abspath to to new video

    """
    # check input file exists
    assert os.path.isfile(input_file), "input video does not exist."
    # check directory for saving outputs exists
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    # pass args as a list (no shell) so paths with spaces work
    ffmpeg_cmd = [FFMPEG, '-i', input_file, '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
                  '-c:a', 'copy', '-y', output_file]
    result = subprocess.run(ffmpeg_cmd, capture_output=True, text=True)
    if result.returncode != 0 or not os.path.isfile(output_file):
        msg = f"ffmpeg failed (code {result.returncode}):"
        raise RuntimeError(msg + os.linesep + result.stderr[-2000:])


using ffmpeg: c:\Users\Almog\anaconda3\envs\DEEPLABCUT\Library\bin\ffmpeg.exe


In [2]:
from pathlib import Path

# --- config --------------------------------------------------------------
INPUT_FOLDER  = r"C:\Maze\Video Data\NDR1"   # folder with the source videos
OUTPUT_FOLDER = None            # None -> write next to each source
VIDEO_EXTS    = (".mp4", ".avi", ".mov", ".mkv", ".m4v")
OVERWRITE     = False           # re-encode even if the output already exists
# ------------------------------------------------------------------------

src_dir = Path(INPUT_FOLDER)
out_dir = Path(OUTPUT_FOLDER) if OUTPUT_FOLDER else src_dir
out_dir.mkdir(parents=True, exist_ok=True)

videos = sorted(
    p for p in src_dir.iterdir()
    if p.suffix.lower() in VIDEO_EXTS and not p.stem.endswith(".reencoded")
)
print(f"Found {len(videos)} video(s) in {src_dir}\n")

for i, src in enumerate(videos, 1):
    dst = out_dir / f"{src.stem}.reencoded.mp4"
    if dst.exists() and not OVERWRITE:
        print(f"[{i}/{len(videos)}] skip (exists): {dst.name}")
        continue
    print(f"[{i}/{len(videos)}] re-encoding {src.name} -> {dst.name}")
    reencode_video(str(src), str(dst))

print("\nDone.")

Found 1 video(s) in C:\Maze\Video Data\NDR1

[1/1] re-encoding NDR1_2026_10_11_14_47_39_stitched_blacked.mp4 -> NDR1_2026_10_11_14_47_39_stitched_blacked.reencoded.mp4

Done.


In [6]:
from collections import Counter
all_files = [p for p in src_dir.iterdir() if p.is_file()]
print("total files:      ", len(all_files))
print("by extension:     ", Counter(p.suffix.lower() for p in all_files))
print("already .reencoded:", sum(p.stem.endswith(".reencoded") for p in all_files))
print("counted as source: ", len(videos))
print("would skip (exists):", sum((out_dir / f'{s.stem}.reencoded.mp4').exists() for s in videos))
print("subdirs:          ", [p.name for p in src_dir.iterdir() if p.is_dir()])

total files:       166
by extension:      Counter({'.identifier': 76, '.mp4': 46, '.pickle': 22, '.h5': 11, '.csv': 11})
already .reencoded: 14
counted as source:  32
would skip (exists): 14
subdirs:           ['plot-poses']
